In [1]:
# ===== Advanced Retrieval Demo (No API Key) =====
# Multi-Query Retrieval + HyDE-style retrieval using simple query generators

!pip -q install -U sentence-transformers chromadb

from sentence_transformers import SentenceTransformer
import chromadb

# 1) Mini corpus (pretend these are chunks from PDFs)
docs = [
    ("c1", "Multi-Query Retrieval creates multiple rewrites of a user query to improve recall in vector search."),
    ("c2", "HyDE generates a hypothetical answer to a question and uses it as a search query to retrieve relevant documents."),
    ("c3", "A re-ranker reorders retrieved chunks using a cross-encoder to improve precision."),
    ("c4", "Vector similarity search can miss relevant context when the query uses different wording than the documents."),
    ("c5", "Query expansion methods like synonyms and paraphrases can increase recall in information retrieval."),
]

# 2) Vector store (Chroma)
embed = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
col = client.get_or_create_collection("adv_retrieval_demo")

ids = [i for i,_ in docs]
texts = [t for _,t in docs]
embs = embed.encode(texts, normalize_embeddings=True).tolist()

try:
    col.delete(ids=ids)
except Exception:
    pass
col.add(ids=ids, documents=texts, embeddings=embs)

def retrieve(query, k=3):
    q_emb = embed.encode([query], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q_emb, n_results=k)
    return list(zip(res["ids"][0], res["documents"][0]))

# 3) Multi-Query generator (simple teaching version)
def multi_queries(user_q: str):
    return [
        user_q,
        f"Explain {user_q} with retrieval and context",
        f"Definition and purpose of {user_q}",
        f"{user_q} technique in RAG systems",
    ]

# 4) HyDE generator (simple teaching version)
def hyde_query(user_q: str):
    # Hypothetical answer (fake, but representative)
    hypo = (
        f"{user_q} is a method used in retrieval-augmented generation to improve recall by "
        "searching with a generated hypothetical answer that resembles what relevant documents contain."
    )
    return hypo

# 5) Advanced retrieval: union + deduplicate
def advanced_retrieve(user_q: str, k_each=3):
    # Multi-Query
    mqs = multi_queries(user_q)
    mq_hits = []
    for q in mqs:
        mq_hits.extend(retrieve(q, k=k_each))

    # HyDE
    hq = hyde_query(user_q)
    hyde_hits = retrieve(hq, k=k_each)

    # Combine + deduplicate by id (keep first occurrence)
    seen = set()
    combined = []
    for item in mq_hits + hyde_hits:
        if item[0] not in seen:
            combined.append(item)
            seen.add(item[0])

    return mqs, hq, combined

# 6) Run demo
question = "HyDE retrieval"
mqs, hq, results = advanced_retrieve(question, k_each=2)

print("USER QUESTION:", question)

print("\n--- Multi-Query variants ---")
for i,q in enumerate(mqs,1):
    print(f"{i}. {q}")

print("\n--- HyDE hypothetical answer used as query ---")
print(hq)

print("\n--- Combined retrieved chunks (deduped) ---")
for i,(cid,txt) in enumerate(results,1):
    print(f"{i}. {cid}: {txt}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

USER QUESTION: HyDE retrieval

--- Multi-Query variants ---
1. HyDE retrieval
2. Explain HyDE retrieval with retrieval and context
3. Definition and purpose of HyDE retrieval
4. HyDE retrieval technique in RAG systems

--- HyDE hypothetical answer used as query ---
HyDE retrieval is a method used in retrieval-augmented generation to improve recall by searching with a generated hypothetical answer that resembles what relevant documents contain.

--- Combined retrieved chunks (deduped) ---
1. c2: HyDE generates a hypothetical answer to a question and uses it as a search query to retrieve relevant documents.
2. c1: Multi-Query Retrieval creates multiple rewrites of a user query to improve recall in vector search.
3. c5: Query expansion methods like synonyms and paraphrases can increase recall in information retrieval.
4. c3: A re-ranker reorders retrieved chunks using a cross-encoder to improve precision.
